In [0]:
from pyspark.sql.functions import col, lower, trim, when, current_timestamp, coalesce, lit

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

print("Silver schema ready")

In [0]:
products = (
    spark.table("azure_blob_storage.src_products")
    .drop("_file", "_line", "_modified", "_fivetran_synced")
    .withColumn("product_name", coalesce(lower(trim(col("product_name"))), lit("none")))
    .withColumn("category", coalesce(lower(trim(col("category"))), lit("none")))
    .withColumn("cost_price", 
        when(col("cost_price").isNull(), 0)
        .when(col("cost_price") < 0, 0)
        .otherwise(col("cost_price"))
    )
    .withColumn("ingestion_ts", current_timestamp())
    .dropDuplicates(["product_id"])
)

products.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.products")
print(f"products: {products.count()} records")

In [0]:
print("\nSample data:")
display(spark.table("silver.products").limit(5))